In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
import matplotlib.pyplot as plt


In [3]:
receita = pd.read_excel('exerc9.xlsx', sheet_name='receita')
fluxo = pd.read_excel('exerc9.xlsx', sheet_name='fluxo')

In [7]:
receita = receita.set_index('Ano')
fluxo = fluxo.set_index('Ano')

In [13]:
ativos = receita.columns
anos = receita.index

In [47]:
dic_receita = receita.T.to_dict()
dic_fluxo = fluxo.T.to_dict()

In [73]:
dic_receita[5]

{'Ativo 1': 2.0, 'Ativo 2': 1.0, 'Ativo 3': 0.5}

In [75]:
model = pyo.ConcreteModel()

model.ativos = pyo.Set(initialize = ativos)
model.anos = pyo.Set(initialize = anos)

model.x = pyo.Var(model.ativos,model.anos, bounds=(0,None), domain=pyo.NonNegativeReals)

def objetivo1(model):

    return sum(model.x[a,ano]*1000000 for a in model.ativos for ano in model.anos)
model.objetivo = pyo.Objective(rule=objetivo1, sense=pyo.minimize)

def rest_fluxo(model, ano):

    return sum(model.x[ativo,ano] * dic_receita[ano][ativo] for ativo in model.ativos ) == dic_fluxo[ano]['Fluxo']
model.rest_fluxo = pyo.Constraint(model.anos, rule=rest_fluxo)


In [77]:
opt = SolverFactory('cplex')
resultado = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmp4f0vahwf.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpswnf9rmt.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpswnf9rmt.pyomo.lp
Objective sense      : Minimize
Variables            :       9
Objective nonzeros   :       9
Linear constraints   :       3  [Equal: 3]
  Nonzeros           :       8
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 1000000.         Max   : 1000000.   

In [76]:
model.pprint()

2 Set Declarations
    anos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {5, 10, 20}
    ativos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'Ativo 1', 'Ativo 2', 'Ativo 3'}

1 Var Declarations
    x : Size=9, Index=ativos*anos
        Key             : Lower : Value : Upper : Fixed : Stale : Domain
         ('Ativo 1', 5) :     0 :  None :  None : False :  True : NonNegativeReals
        ('Ativo 1', 10) :     0 :  None :  None : False :  True : NonNegativeReals
        ('Ativo 1', 20) :     0 :  None :  None : False :  True : NonNegativeReals
         ('Ativo 2', 5) :     0 :  None :  None : False :  True : NonNegativeReals
        ('Ativo 2', 10) :     0 :  None :  None : False :  True : NonNegativeReals
        ('Ativo 2', 20) :     0 :  None :  None : False :  True : NonNegativeReals
         ('Ativo 3', 5) :     0 :  Non

In [79]:
for i,j in model.x:
    print(f'{i} - Ano {j}: ', pyo.value(model.x[i,j]))

print('Objetivo: ',pyo.value(model.objetivo))

Ativo 1 - Ano 5:  200.0
Ativo 1 - Ano 10:  0.0
Ativo 1 - Ano 20:  0.0
Ativo 2 - Ano 5:  0.0
Ativo 2 - Ano 10:  0.0
Ativo 2 - Ano 20:  0.0
Ativo 3 - Ano 5:  0.0
Ativo 3 - Ano 10:  100.0
Ativo 3 - Ano 20:  150.0
Objetivo:  450000000.0
